# Stable Diffusion — Forge (AUTOMATIC1111)
Maintained fork of [TheLastBen/fast-stable-diffusion](https://github.com/TheLastBen/fast-stable-diffusion).  
Targets [lllyasviel/stable-diffusion-webui-forge](https://github.com/lllyasviel/stable-diffusion-webui-forge).  
**Requires a GPU runtime** — Runtime → Change runtime type → T4 GPU.

In [ ]:
#@title # 1 · Connect Google Drive
from google.colab import drive
from IPython.display import clear_output
import ipywidgets as widgets
import os

def btn(msg, style='primary', width='auto'):
    b = widgets.Button(description=msg, disabled=True,
                       button_style=style,
                       layout=widgets.Layout(width=width))
    display(b)

Shared_Drive = '' #@param {type:"string"}
#@markdown Leave empty if you are not using a Shared Drive.

drive.mount('/content/gdrive')

if Shared_Drive and os.path.exists('/content/gdrive/Shareddrives'):
    DRIVE_PATH = f'Shareddrives/{Shared_Drive}'
else:
    DRIVE_PATH = 'MyDrive'

SD_ROOT   = f'/content/gdrive/{DRIVE_PATH}/sd'
FORGE_DIR = f'{SD_ROOT}/stable-diffusion-webui-forge'
os.makedirs(SD_ROOT, exist_ok=True)
clear_output()
btn('\u2714 Drive connected', 'success')

In [ ]:
#@title # 2 · Install Forge + dependencies
import os, sys, subprocess, shutil
from IPython.display import clear_output

# ── versioned blob URL — updated automatically by fast-stable-diffusion-deps CI ──
DEPS_URL = "https://github.com/danielef/fast-stable-diffusion-deps/releases/download/v1.0.7/gcolabdeps_forge.tar.zst"

DEPS_DIR = '/content/forge_deps'
os.makedirs(DEPS_DIR, exist_ok=True)

# ── 1. Clone or update Forge ──────────────────────────────────────
if not os.path.exists(FORGE_DIR):
    print('Cloning Forge…')
    !git clone -q --depth 1 https://github.com/lllyasviel/stable-diffusion-webui-forge "{FORGE_DIR}"
else:
    print('Updating Forge…')
    !git -C "{FORGE_DIR}" pull -q --ff-only

# ── 2. Install pinned dependency blob ─────────────────────────────
print('Installing dependencies…')
if not shutil.which('zstd'):
    subprocess.run(['apt-get', 'install', '-y', '-q', 'zstd'], check=True, capture_output=True)
!wget -q -O /tmp/gcolabdeps_forge.tar.zst "{DEPS_URL}"
subprocess.run(
    ['tar', '-C', DEPS_DIR, '--zstd', '-xf', '/tmp/gcolabdeps_forge.tar.zst'],
    check=True,
)
os.remove('/tmp/gcolabdeps_forge.tar.zst')

# Hard stop: verify the critical package landed in DEPS_DIR.
if not os.path.exists(f'{DEPS_DIR}/pytorch_lightning'):
    raise RuntimeError(
        f'pytorch_lightning missing from {DEPS_DIR} after extraction — '
        f'the blob may not contain it. Check {DEPS_URL}'
    )

# sys.path for current kernel.
if DEPS_DIR not in sys.path:
    sys.path.insert(0, DEPS_DIR)

# .pth file — Python reads this at startup for every new process.
PY_VER = f'python{sys.version_info.major}.{sys.version_info.minor}'
_dist  = f'/usr/local/lib/{PY_VER}/dist-packages'
try:
    os.makedirs(_dist, exist_ok=True)
    with open(f'{_dist}/forge_deps.pth', 'w') as _f:
        _f.write(DEPS_DIR + '\n')
except PermissionError:
    pass

# ── 3. Environment ────────────────────────────────────────────────
os.environ['TRANSFORMERS_CACHE'] = f'{FORGE_DIR}/cache'
os.environ['TORCH_HOME']         = f'{FORGE_DIR}/cache'
os.environ['PYTHONWARNINGS']     = 'ignore'
os.makedirs(f'{FORGE_DIR}/cache', exist_ok=True)

clear_output()
btn('✔ Done', 'success')

In [ ]:
#@title # 3 · Load model
import os, requests
from IPython.display import clear_output
from urllib.request import urlopen, Request
from urllib.parse import urlparse, parse_qs, unquote
from tqdm import tqdm
import gdown

Use_Temp_Storage = False #@param {type:"boolean"}
#@markdown Store models in `/content` (lost after session) instead of Google Drive.

#@markdown ---
Model_Version = 'SDXL' #@param ["SDXL", "SD 1.5", "SD 1.5 Inpainting", "SD 2.1-768"]
#@markdown **Or** provide a path to a model or folder already on Drive:
PATH_to_MODEL = '' #@param {type:"string"}
#@markdown **Or** paste a direct download URL (HuggingFace, CivitAI, Google Drive):
MODEL_URL = '' #@param {type:"string"}

MODELS_DIR = ('/content/temp_models' if Use_Temp_Storage
              else f'{FORGE_DIR}/models/Stable-diffusion')
os.makedirs(MODELS_DIR, exist_ok=True)


def _url_source(url):
    host = urlparse(url).netloc
    if 'civitai.com'      in host: return 'civitai'
    if 'drive.google.com' in host: return 'gdrive'
    if 'huggingface.co'   in host: return 'huggingface'
    return 'direct'


def _filename_from_redirect(url):
    """Resolve the real filename from a CivitAI-style redirect."""
    try:
        r = requests.get(url, allow_redirects=False, timeout=10)
        qs = parse_qs(urlparse(r.headers.get('Location', '')).query)
        disp = qs.get('response-content-disposition', [''])[0]
        for part in disp.split(';'):
            if part.strip().startswith('filename='):
                return unquote(part.split('=', 1)[1].strip().strip('"'))
    except Exception:
        pass
    return os.path.basename(urlparse(url).path) or 'model.safetensors'


def _download(url, dst, label='Downloading'):
    req = Request(url, headers={'User-Agent': 'torch.hub'})
    with urlopen(req) as u:
        total = int(u.info().get('Content-Length') or 0) or None
    with tqdm(total=total, unit='B', unit_scale=True,
              bar_format=f'{label} |{{bar:20}}| {{percentage:3.0f}}%') as pbar:
        req2 = Request(url, headers={'User-Agent': 'torch.hub'})
        with urlopen(req2) as u, open(dst, 'wb') as f:
            while chunk := u.read(65536):
                f.write(chunk)
                pbar.update(len(chunk))


PRESET_MODELS = {
    'SDXL': (
        'sd_xl_base_1.0.safetensors',
        'https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors',
    ),
    'SD 1.5': (
        'v1-5-pruned-emaonly.safetensors',
        'https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors',
    ),
    'SD 1.5 Inpainting': (
        'sd-v1-5-inpainting.safetensors',
        'https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-inpainting/resolve/main/sd-v1-5-inpainting.safetensors',
    ),
    'SD 2.1-768': (
        'v2-1_768-ema-pruned.safetensors',
        'https://huggingface.co/stabilityai/stable-diffusion-2-1/resolve/main/v2-1_768-ema-pruned.safetensors',
    ),
}

# ── Resolve MODEL path ────────────────────────────────────────────
if PATH_to_MODEL:
    if not os.path.exists(PATH_to_MODEL):
        btn(f'\u2718 Path not found: {PATH_to_MODEL}', 'danger')
        raise FileNotFoundError(PATH_to_MODEL)
    MODEL = PATH_to_MODEL
    btn('\u2714 Using model from Drive', 'success')

elif MODEL_URL:
    src  = _url_source(MODEL_URL)
    name = (_filename_from_redirect(MODEL_URL) if src != 'gdrive'
            else os.path.basename(urlparse(MODEL_URL).path) or 'model.safetensors')
    MODEL = os.path.join(MODELS_DIR, name)
    if not os.path.exists(MODEL):
        if src == 'gdrive':
            gdown.download(url=MODEL_URL, output=MODEL, quiet=False, fuzzy=True)
        else:
            _download(MODEL_URL, MODEL, f'Downloading {name}')
        clear_output()
    btn('\u2714 Model ready', 'success')

else:
    fname, link = PRESET_MODELS[Model_Version]
    MODEL = os.path.join(MODELS_DIR, fname)
    if not os.path.exists(MODEL):
        _download(link, MODEL, f'Downloading {fname}')
        clear_output()
    btn('\u2714 Model ready', 'success')

In [ ]:
#@title # 4 · Download LoRA  (optional)
LoRA_URL = '' #@param {type:"string"}
#@markdown HuggingFace, CivitAI, or Google Drive link. Leave empty to skip.

if not LoRA_URL.strip():
    btn('\u2714 Skipped', 'primary')
else:
    LORA_DIR = f'{FORGE_DIR}/models/Lora'
    os.makedirs(LORA_DIR, exist_ok=True)

    src  = _url_source(LoRA_URL)
    name = (_filename_from_redirect(LoRA_URL) if src != 'gdrive'
            else os.path.basename(urlparse(LoRA_URL).path) or 'lora.safetensors')
    dst  = os.path.join(LORA_DIR, name)

    if not os.path.exists(dst):
        if src == 'gdrive':
            gdown.download(url=LoRA_URL, output=dst, quiet=False, fuzzy=True)
        else:
            _download(LoRA_URL, dst, f'Downloading {name}')
        clear_output()

    btn('\u2714 LoRA ready', 'success')

In [ ]:
#@title # 5 · Launch
import os, subprocess, re
from IPython.display import display, HTML

# ── Cloudflare Tunnel (free, no account needed) ───────────────────
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
        -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared


def _start_tunnel(port=7860):
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            return m.group(0), proc
    return None, proc


UI_User     = '' #@param {type:"string"}
UI_Password = '' #@param {type:"string"}
#@markdown Add Gradio credentials (optional). Leave both empty to skip.

auth_flag = (f'--gradio-auth {UI_User}:{UI_Password}'
             if UI_User and UI_Password else '')

model_flag = (f'--ckpt "{MODEL}"' if os.path.isfile(MODEL)
              else f'--ckpt-dir "{MODEL}"')

launch_args = (
    '--api '
    '--no-download-sd-model '
    '--no-half-vae '
    '--disable-safe-unpickle '
    '--enable-insecure-extension-access '
    '--disable-console-progressbars '
    '--skip-version-check '
    f'{model_flag} '
    f'{auth_flag}'
)

print('Starting tunnel…')
url, _tunnel = _start_tunnel(7860)
if url:
    display(HTML(f'<h3>&#127760; <a href="{url}" target="_blank">{url}</a></h3>'))
else:
    print('Tunnel URL not detected — check output above.')

os.chdir(FORGE_DIR)
# PYTHONPATH set at shell level — guaranteed to reach the Python subprocess.
!PYTHONPATH="/content/forge_deps" python webui.py {launch_args}